# 📦 Supply Chain Performance Analytics
## Part 1 — Exploration & Data Cleaning

**Author:** Kawtar Barouti  
**Dataset:** DataCo Smart Supply Chain (180K+ orders)  
**Objective:** Identify root causes of late deliveries and optimize logistics performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded ✓')

## 1. Data Loading & First Look

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df.head()

In [ ]:
# Column overview
print('=== DATA TYPES ===')
print(df.dtypes.value_counts())
print(f'\n=== NULL VALUES (top 10) ===')
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls > 0].head(10))

In [ ]:
# Key columns for our analysis
key_cols = [
    'order date (DateOrders)', 'shipping date (DateOrders)',
    'Delivery Status', 'Late_delivery_risk', 'Shipping Mode',
    'Customer Segment', 'Order Item Quantity', 'Sales per customer',
    'Order Profit Per Order', 'Order Region', 'Market',
    'Category Name', 'Order Item Discount Rate', 'Order Item Profit Ratio'
]

print(f'Key columns: {len(key_cols)}')
df[key_cols].describe()

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Parse dates
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping_date'] = pd.to_datetime(df['shipping date (DateOrders)'])

# Calculate delivery delay in days
df['delivery_days'] = (df['shipping_date'] - df['order_date']).dt.days

# Create order size buckets
df['order_size'] = pd.cut(
    df['Order Item Quantity'],
    bins=[0, 2, 4, 100],
    labels=['Small (1-2)', 'Medium (3-4)', 'Large (5+)']
)

# Extract time features
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['order_year'] = df['order_date'].dt.year
df['order_quarter'] = df['order_date'].dt.quarter

print(f'Delivery days — min: {df["delivery_days"].min()}, max: {df["delivery_days"].max()}, mean: {df["delivery_days"].mean():.1f}')
print(f'Date range: {df["order_date"].min()} → {df["order_date"].max()}')
print('\nFeature engineering complete ✓')

## 3. Key Metrics Overview

In [ ]:
total_orders = len(df)
late_orders = df['Late_delivery_risk'].sum()
late_rate = late_orders / total_orders * 100
avg_delivery = df['delivery_days'].mean()
total_revenue = df['Sales per customer'].sum()

print('=' * 50)
print('         SUPPLY CHAIN KPI OVERVIEW')
print('=' * 50)
print(f'  Total Orders:        {total_orders:>12,}')
print(f'  Late Deliveries:     {late_orders:>12,}')
print(f'  Late Delivery Rate:  {late_rate:>11.1f}%')
print(f'  Avg Delivery Days:   {avg_delivery:>11.1f}')
print(f'  Total Revenue:       ${total_revenue:>11,.0f}')
print('=' * 50)

## 4. Load into SQLite for SQL Analysis

In [ ]:
# Create SQLite database
conn = sqlite3.connect('../data/supply_chain.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

# Verify
result = pd.read_sql('SELECT COUNT(*) as row_count FROM orders', conn)
print(f'Loaded {result["row_count"].iloc[0]:,} rows into supply_chain.db ✓')
conn.close()

## 5. Quick Exploration Charts

In [ ]:
# Distribution of delivery status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delivery status distribution
status_counts = df['Delivery Status'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
axes[0].barh(status_counts.index, status_counts.values, color=colors[:len(status_counts)])
axes[0].set_title('Delivery Status Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Orders')

# Shipping mode distribution
mode_counts = df['Shipping Mode'].value_counts()
axes[1].barh(mode_counts.index, mode_counts.values, color='#2E75B6')
axes[1].set_title('Orders by Shipping Mode', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Orders')

plt.tight_layout()
plt.savefig('../visuals/delivery_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → visuals/delivery_overview.png ✓')

In [ ]:
# Delivery days distribution
fig, ax = plt.subplots(figsize=(12, 5))
df['delivery_days'].hist(bins=50, color='#2E75B6', edgecolor='white', ax=ax)
ax.axvline(df['delivery_days'].mean(), color='#C0392B', linestyle='--', linewidth=2, label=f'Mean: {df["delivery_days"].mean():.1f} days')
ax.set_title('Distribution of Delivery Time (Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Days')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('../visuals/delivery_days_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Next:** Open `02_visualizations.ipynb` for deep-dive charts and business insights.